In [8]:
import kagglehub

# Download latest version
path = kagglehub.competition_download('week-03-hidden-pairs')

print("Path to competition files:", path)

Path to competition files: /kaggle/input/competitions/week-03-hidden-pairs


In [12]:
!pip install cleanlab

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 306.1/306.1 kB 7.2 MB/s eta 0:00:00ta 0:00:01


In [1]:
import pandas as pd
import random
import numpy as np
import torch
import matplotlib.pyplot as plt
import seaborn as sns
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import roc_auc_score
from pathlib import Path
from tqdm.auto import tqdm
import time
import cleanlab

/Users/platon/Documents/sbmt_kaggle/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def set_seed(seed=42):
    """Фиксирует seed для random, NumPy и PyTorch, включая CUDA."""

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    torch.use_deterministic_algorithms(True, warn_only=True)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

    print(f'setted seed for {seed}')

In [5]:
SEED = 42
set_seed(SEED)

LOCAL = True

setted seed for 42


In [6]:
DATA_DIR = Path('./data') if LOCAL else Path('/kaggle/input/competitions/week-03-hidden-pairs')

train_df: pd.DataFrame = pd.read_csv(DATA_DIR / 'train.csv')
val_df: pd.DataFrame = pd.read_csv(DATA_DIR / 'validation.csv')
test_df: pd.DataFrame = pd.read_csv(DATA_DIR / 'test.csv')
sample_submission: pd.DataFrame = pd.read_csv(DATA_DIR / 'sample_submission.csv')

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)
print("sample submission:", sample_submission.shape)

train: (2520, 514)
val: (560, 514)
test: (1116, 513)
sample submission: (1116, 2)


In [7]:
assert train_df.isna().sum().sum() == 0, "train_df contains NaN values"
assert val_df.isna().sum().sum() == 0, "val_df contains NaN values"
assert test_df.isna().sum().sum() == 0, "test_df contains NaN values"

assert train_df['row_id'].is_unique, "train_df contains duplicate ids"
assert val_df['row_id'].is_unique, "val_df contains duplicate ids"
assert test_df['row_id'].is_unique, "test_df contains duplicate ids"

In [8]:
feature_columns: list[str] = [
    col for col in train_df.columns if col not in ['row_id', 'target']
]

X_train: pd.DataFrame = train_df[feature_columns]
y_train: pd.Series = train_df['target']

X_val: pd.DataFrame = val_df[feature_columns]
y_val: pd.Series = val_df['target']

X_test: pd.DataFrame = test_df[feature_columns]

print('# of features:', len(feature_columns))
print('X_train shape:', X_train.shape)
print('y_train shape:', y_train.shape)
print('X_val shape:', X_val.shape)
print('y_val shape:', y_val.shape)
print('X_test shape:', X_test.shape)

# of features: 512
X_train shape: (2520, 512)
y_train shape: (2520,)
X_val shape: (560, 512)
y_val shape: (560,)
X_test shape: (1116, 512)


In [ ]:
def _extract_auc_history(model, X_train, y_train, X_val, y_val):
    """Возвращает списки train/val ROC AUC по итерациям бустинга.

    Для дерева и случайного леса возвращает два пустых списка.
    """
    if isinstance(model, CatBoostClassifier):
        history = model.get_evals_result()
        # Два eval_set при обучении: сначала train, затем validation.
        return history['validation_0']['AUC'], history['validation_1']['AUC']

    if isinstance(model, LGBMClassifier):
        history = model.evals_result_
        return history['train']['auc'], history['val']['auc']

    if isinstance(model, GradientBoostingClassifier):
        train_scores = [
            roc_auc_score(y_train, probs[:, 1])
            for probs in model.staged_predict_proba(X_train)
        ]
        val_scores = [
            roc_auc_score(y_val, probs[:, 1])
            for probs in model.staged_predict_proba(X_val)
        ]
        return train_scores, val_scores

    return [], []


def _fit_single_model(model, X_train, y_train, X_val, y_val):
    """Обучает копию модели и возвращает вероятности, AUC, время fit и кривые.

    Validation используется для оценки; CatBoost сохраняет все итерации.
    Время fit не включает предсказания и вычисление истории AUC.
    """
    fitted_model = clone(model)
    start_time = time.perf_counter()

    if isinstance(fitted_model, CatBoostClassifier):
        fitted_model.set_params(eval_metric='AUC')
        fitted_model.fit(
            X_train, y_train,
            eval_set=[(X_train, y_train), (X_val, y_val)],
            use_best_model=False, verbose=False,
        )
    elif isinstance(fitted_model, LGBMClassifier):
        fitted_model.fit(
            X_train, y_train,
            eval_set=[(X_train, y_train), (X_val, y_val)],
            eval_names=['train', 'val'], eval_metric='auc',
        )
    else:
        fitted_model.fit(X_train, y_train)

    fit_time = time.perf_counter() - start_time
    train_probs = fitted_model.predict_proba(X_train)[:, 1]
    val_probs = fitted_model.predict_proba(X_val)[:, 1]
    train_history, val_history = _extract_auc_history(
        fitted_model, X_train, y_train, X_val, y_val,
    )

    return {
        'val_probs': val_probs,
        'train_auc': roc_auc_score(y_train, train_probs),
        'val_auc': roc_auc_score(y_val, val_probs),
        'fit_time': fit_time,
        'train_history': train_history,
        'val_history': val_history,
    }


def _plot_learning_curves(histories, ncols=2):
    """Рисует train/val ROC AUC по итерациям для моделей с историей обучения."""
    histories = {name: result for name, result in histories.items() if result['val_history']}
    if not histories:
        return

    sns.set_theme(style='darkgrid')
    nrows = (len(histories) + ncols - 1) // ncols
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(6 * ncols, 4 * nrows), squeeze=False,
    )
    axes = axes.ravel()

    for ax, (name, result) in zip(axes, histories.items()):
        for key, label in [('train_history', 'Train AUC'), ('val_history', 'Val AUC')]:
            scores = result[key]
            ax.plot(range(1, len(scores) + 1), scores, label=label)
        ax.set(title=name, xlabel='Iteration', ylabel='ROC AUC')
        ax.legend()

    # Удаление неиспользуемых ячеек сетки (если количество моделей нечетное)
    for ax in axes[len(histories):]:
        fig.delaxes(ax)

    fig.tight_layout()
    plt.show()


def run_model_pipeline(models_dict, X_train, y_train, X_val, y_val, plot_curves=True):
    """Обучает модели и возвращает вероятности на val и таблицу AUC/времени.

    Строки вероятностей соответствуют X_val; метрики отсортированы по Val AUC.
    """
    val_predictions = {}
    metrics_list = []
    histories = {}

    for name, model in models_dict.items():
        result = _fit_single_model(model, X_train, y_train, X_val, y_val)
        val_predictions[name] = result['val_probs']
        histories[name] = result
        metrics_list.append({
            'Model': name,
            'Train AUC': result['train_auc'],
            'Val AUC': result['val_auc'],
            'Fit Time (s)': round(result['fit_time'], 3),
        })

    preds_df = pd.DataFrame(val_predictions, index=X_val.index)
    metrics_df = pd.DataFrame(metrics_list).sort_values('Val AUC', ascending=False).reset_index(drop=True)
    if plot_curves:
        _plot_learning_curves(histories)
    return preds_df, metrics_df


def run_model_pipeline_search(
    models_dict, param_grids, X_train, y_train, X_val, y_val,
    n_iter=10, cv=5, scoring='roc_auc', verbose=10, n_jobs=-1, plot_curves=True,
):
    """Подбирает параметры на train для моделей, перечисленных в param_grids.

    Лучшую конфигурацию обучает на всём train один раз и оценивает на val.
    Возвращает вероятности и метрики: CV Score/STD относятся к scoring,
    Train/Val AUC — к финальной модели, Fit Time — к её обучению без поиска.
    """
    val_predictions = {}
    metrics_list = []
    histories = {}

    for name, param_grid in param_grids.items():
        model = models_dict[name]
        search = RandomizedSearchCV(
            model, param_distributions=param_grid, n_iter=n_iter,
            cv=cv, scoring=scoring, random_state=SEED,
            verbose=verbose, n_jobs=n_jobs, refit=False,
        )
        search.fit(X_train, y_train)
        best_model = clone(model).set_params(**search.best_params_)
        result = _fit_single_model(best_model, X_train, y_train, X_val, y_val)

        val_predictions[name] = result['val_probs']
        histories[name] = result
        metrics_list.append({
            'Model': name,
            'Best Params': search.best_params_,
            'CV Score': search.best_score_,
            'CV STD': search.cv_results_['std_test_score'][search.best_index_],
            'Train AUC': result['train_auc'],
            'Val AUC': result['val_auc'],
            'Fit Time (s)': round(result['fit_time'], 3),
        })

    preds_df = pd.DataFrame(val_predictions, index=X_val.index)
    metrics_df = pd.DataFrame(metrics_list).sort_values('Val AUC', ascending=False).reset_index(drop=True)
    if plot_curves:
        _plot_learning_curves(histories)
    return preds_df, metrics_df


In [10]:
models = {
    'Decision Tree': DecisionTreeClassifier(max_depth=5, random_state=SEED),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=5, random_state=SEED, n_jobs=-1),
    'GBDT 3': GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=SEED),
    'GBDT 5': GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=5, random_state=SEED),
    'GBDT 2': GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=2, random_state=SEED),
    'LightGBM': LGBMClassifier(n_estimators=100, learning_rate=0.05, max_depth=3, metric='auc', random_state=SEED, verbose=-1),
    'CatBoost': CatBoostClassifier(iterations=100, learning_rate=0.05, depth=4, eval_metric='AUC', random_seed=SEED)
}

In [ ]:
preds_df, metrics_df = run_model_pipeline(models, X_train, y_train, X_val, y_val)

In [ ]:
metrics_df

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
param_grids = {
    'Random Forest': {
        'n_estimators': [100, 200, 300, 500],
        'max_depth': [3, 5, 7, 10, None],
        'min_samples_leaf': [1, 2, 5, 10],
        'max_features': ['sqrt', 'log2', 0.5],
    },
}

In [ ]:
# preds_df_search, metrics_df_search = run_model_pipeline_search(models, param_grids, X_train, y_train, X_val, y_val, n_iter=30, cv=cv, scoring='roc_auc', plot_curves=True, verbose=10, n_jobs=-1)

In [ ]:
preds_df_search

In [ ]:
metrics_df_search.to_clipboard(index=False)
metrics_df_search

In [ ]:
model = RandomForestClassifier(n_estimators=200, max_depth=5, min_samples_split=50, min_samples_leaf=20, random_state=SEED, n_jobs=-1)
cl = cleanlab.classification.CleanLearning(model, seed=SEED, verbose=True)
cl.fit(X_train, y_train)
model = clone(model)
model.fit(X_train, y_train)

# issues = cl.get_label_issues()
# print(issues['is_label_issue'].sum())
print("Val AUC (cleaned):", roc_auc_score(y_val, cl.predict_proba(X_val)[:, 1]))
print('Val AUC (model):', roc_auc_score(y_val, model.predict_proba(X_val)[:, 1]))
print('model params:', model.get_params())